# LR2.2 — Week 9 hold-out evaluation

**Added by Sihao Cui (Role 4) during report consolidation, 15 October 2026.**

The notebook supplied with the LR2.2 section (`lr22_goals_scored_luke.ipynb`) evaluates the
model by 10-fold cross-validation and reports RMSE only. The report section, however, quotes
the Week 9 practical protocol: an 80/20 train/test split, the five standard performance
metrics, and a comparison against a baseline that predicts the mean of the training set.

This notebook supplies that protocol so the quoted figures can be reproduced from the
repository. It reproduces them exactly: MAE 0.918, MSE 1.637, RMSE 1.280, normalised RMSE
0.183, R² 0.184.

The same computation is available as a script for both models in `src/evaluate_week9.py`.

In [1]:
import math
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

RANDOM_STATE = 42
TEST_SIZE = 0.2

ALL_EIGHT = ["team_gf_pg", "opp_ga_pg", "opp_gf_pg", "team_log_rank",
             "opp_log_rank", "team_log_value", "is_host_home", "is_knockout"]
RETAINED_SIX = ["team_gf_pg", "opp_ga_pg", "team_log_rank",
                "opp_log_rank", "team_log_value", "is_host_home"]

df = pd.read_csv("../data/lr22_team_match_208.csv")
print("rows:", len(df), "| target: goals_scored")
print("mean %.3f | variance %.3f | zero rate %.1f%%"
      % (df.goals_scored.mean(), df.goals_scored.var(),
         100 * (df.goals_scored == 0).mean()))

rows: 208 | target: goals_scored
mean 1.481 | variance 1.942 | zero rate 26.4%


## Hold-out protocol

Following `multiple_linreg.py` from the Week 9 practical: split 80/20, fit with
scikit-learn's `LinearRegression`, and report MAE, MSE, RMSE, normalised RMSE and R²
against a baseline that always predicts the mean of the training set.

Normalised RMSE divides by the full range of the response, as in the practical.

The reduced model is the six-variable specification retained by the subset search in the
LR2.2 notebook; `opp_gf_pg` and `is_knockout` were dropped there as uninformative.

In [2]:
def holdout(features, label):
    x, y = df[features].values, df["goals_scored"].values
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    model = LinearRegression().fit(x_train, y_train)
    y_pred = model.predict(x_test)
    y_base = [y_train.mean()] * len(y_test)
    y_range = y.max() - y.min()

    def row(true, pred):
        mse = metrics.mean_squared_error(true, pred)
        rmse = math.sqrt(mse)
        return (metrics.mean_absolute_error(true, pred), mse, rmse,
                rmse / y_range, metrics.r2_score(true, pred))

    print(f"{label}  (k={len(features)}, train={len(y_train)}, test={len(y_test)})")
    print(f"{'':22}{'MAE':>9}{'MSE':>9}{'RMSE':>9}{'NRMSE':>9}{'R2':>9}")
    print(f"{'linear regression':22}" + "".join(f"{v:>9.3f}" for v in row(y_test, y_pred)))
    print(f"{'training-set mean':22}" + "".join(f"{v:>9.3f}" for v in row(y_test, y_base)))
    eq = f"goals_scored = {model.intercept_:.3f}" + "".join(
        f" {'+' if c >= 0 else '-'} {abs(c):.3f} x {f}"
        for c, f in zip(model.coef_, features))
    print("\n" + eq)
    return model

m6 = holdout(RETAINED_SIX, "Reduced model, six variables retained by the subset search")

Reduced model, six variables retained by the subset search  (k=6, train=166, test=42)
                            MAE      MSE     RMSE    NRMSE       R2
linear regression         0.918    1.637    1.280    0.183    0.184
training-set mean         1.070    2.011    1.418    0.203   -0.002

goals_scored = -1.422 - 0.353 x team_gf_pg + 0.671 x opp_ga_pg - 0.173 x team_log_rank + 0.239 x opp_log_rank + 0.430 x team_log_value + 0.829 x is_host_home


## All eight variables, for comparison

The brief requires the dataset to contain exactly eight explanatory variables. The reduced
model drops two of them, so the full specification is evaluated here on the same split.

In [3]:
m8 = holdout(ALL_EIGHT, "Full model, all eight explanatory variables")

Full model, all eight explanatory variables  (k=8, train=166, test=42)
                            MAE      MSE     RMSE    NRMSE       R2
linear regression         0.923    1.656    1.287    0.184    0.175
training-set mean         1.070    2.011    1.418    0.203   -0.002

goals_scored = -1.511 - 0.372 x team_gf_pg + 0.692 x opp_ga_pg + 0.022 x opp_gf_pg - 0.167 x team_log_rank + 0.250 x opp_log_rank + 0.428 x team_log_value + 0.827 x is_host_home + 0.081 x is_knockout


## Result

Both specifications beat the naive baseline comfortably: RMSE falls from 1.418 to 1.280
(reduced) and 1.287 (full), and R² rises from −0.002 to 0.184 and 0.175. Dropping
`opp_gf_pg` and `is_knockout` costs nothing and gains a little, consistent with their
near-zero correlations with the response (see `src/compare_models_lr22.py`).

The figures from the reduced model are the ones quoted in section 4 of the group report.